In [46]:
import pandas as pd
import numpy as np
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.common.exceptions import TimeoutException, NoSuchElementException 
from selenium.webdriver.common.keys import Keys
from selenium.webdriver.common.action_chains import ActionChains
import re
import time

In [47]:
def set_up_driver():
    chrome_options = Options()
    chrome_options.add_argument("--headless") 
    chrome_options.add_argument("--disable-gpu")
    chrome_options.add_argument("start-maximized")
    chrome_options.add_argument("--disable-blink-features=AutomationControlled") # Tắt tính năng phát hiện tự động của trình duyệt
    chrome_options.add_experimental_option("excludeSwitches", ["enable-automation"]) # Loại bỏ thông báo "Chrome is being controlled by automated test software"
    chrome_options.add_experimental_option('useAutomationExtension', False) # Vô hiệu hóa tiện ích mở rộng tự động hóa
    driver = webdriver.Chrome(options=chrome_options)
    return driver

In [48]:
def get_text(driver, selector):
    try:
        el = driver.find_element(By.CSS_SELECTOR, selector)
        return el.text.strip()
    except:
        return None

In [49]:
def get_section_detail(driver, title):
    try:
        
        # Tìm container chứa nội dung
        container = driver.find_element(By.CSS_SELECTOR, "div.render-html.text-14")
        
        # Lấy tất cả h3 trong container
        h3s = container.find_elements(By.TAG_NAME, "h3")
        title_norm = title.lower().strip()
        
        for h3 in h3s:
            h3_text = h3.get_attribute("textContent") or ""
            
            if title_norm in h3_text.lower():
                # Lấy siblings trực tiếp từ h3
                siblings = h3.find_elements(By.XPATH, "./following-sibling::*")
                
                content = []
                for sib in siblings:
                    tag = sib.tag_name.lower()
                    
                    # Dừng khi gặp h3 tiếp theo
                    if tag == "h3":
                        break
                    
                    # Lấy text
                    t = sib.get_attribute("textContent") or ""
                    t = t.strip()
                    if t:
                        content.append(t)
                
                return "\n".join(content) if content else None
                
    except:
        pass
    return None

In [50]:
def get_product_images(driver):
    images = []

    try:
        img_elements = driver.find_elements(By.CSS_SELECTOR, "img[src*='Products/Images'], img[src*='cdn.tgdd.vn']")

        for img in img_elements:
            src = img.get_attribute("src")
            if src and "Products/Images"in src:
                if "https://cdn.tgdd.vn" in src:
                    match = re.search(r'(https://cdn\.tgdd\.vn/Products/Images/[^\s]+)', src)
                    if match:
                        original_url = match.group(1)
                        if original_url not in images:
                            images.append(original_url)
                elif src not in images:
                    images.append(src)
    except Exception as e:
        print(f"Error {e}")
        pass
    return images

In [51]:
# Lấy thông tin từ bảng grid
def get_product_infomation(driver, label):
    try:
        rows = driver.find_elements(By.CSS_SELECTOR, "#contentRef .grid.grid-cols-3")
        for row in rows:
            label_el = row.find_element(By.CSS_SELECTOR, "div.text-13.font-bold")
            label_text = label_el.get_attribute("textContent") or ""
            
            if label.lower() in label_text.lower():
                value_el = row.find_element(By.CSS_SELECTOR, "div.col-span-2 div.mr-2")
                value_text = value_el.get_attribute("textContent") or ""
                return value_text.strip()
    except Exception as e:
        print(f"Get product information error: {e}")
        pass
    return None

In [52]:
def get_medicine_data(driver):
    try:
        WebDriverWait(driver, 10).until(EC.presence_of_element_located((By.CSS_SELECTOR, "div[class*='text-20'][class*='font-bold']")))
    except:
        pass

    name = get_text(driver, "div.text-20.font-bold")
    if not name:
        try:
            name = driver.title.split(' - ')[0].strip()
        except:
            name = None

    price = None
    try:
        price_el = driver.find_element(By.CSS_SELECTOR, "body > div.bg-secondary-300 > div.w-1200px.mx-auto.min-h-screen.relative.z-1 > div > div.ml-4.flex.h-auto.w-\[540px\].cursor-default.flex-col.gap-20px > div:nth-child(1) > div:nth-child(4) > div > p")
        price = price_el.text.strip()
    except:
        pass
    

    images = get_product_images(driver)


    expand_button_1 = driver.find_element(By.XPATH, "//*[@id='contentRef']/div/div[6]/button")
    expand_button_1.click()
    time.sleep(2)

    cong_dung_tom_tat = get_product_infomation(driver, "Công dụng")
    thanh_phan_chinh = get_product_infomation(driver, "Thành phần chính")
    doi_tuong = get_product_infomation(driver, "Đối tượng sử dụng")
    thuong_hieu = get_product_infomation(driver, "Thương hiệu")
    noi_san_xuat = get_product_infomation(driver, "Nơi sản xuất")
    dang_bao_che = get_product_infomation(driver, "Dạng bào chế")
    han_dung = get_product_infomation(driver, "Hạn dùng")


    close_btn = driver.find_element(By.CSS_SELECTOR, "button[class*='absolute'][class*='right']")
    close_btn.click()
    time.sleep(2)



    expand_button_2 = driver.find_element(By.XPATH, "//*[@id='contentRef']/div/div[3]/button")
    expand_button_2.click()
    time.sleep(2)

    thanh_phan_chi_tiet = get_section_detail(driver, "1. Thành phần")
    cong_dung_chi_tiet = get_section_detail(driver, "2. Công dụng")
    cach_dung = get_section_detail(driver, "3. Cách dùng")
    chong_chi_dinh = get_section_detail(driver, "4. Chống chỉ định")
    tac_dung_phu = get_section_detail(driver, "5. Tác dụng phụ")
    luu_y = get_section_detail(driver, "6. Lưu ý")

    return {
        "ten_thuoc": name,
        "gia": price,
        "hinh_anh": "|".join(images) if images else None,
        "cong_dung_tom_tat": cong_dung_tom_tat,
        "thanh_phan_chinh": thanh_phan_chinh,
        "doi_tuong": doi_tuong,
        "thuong_hieu": thuong_hieu,
        "noi_san_xuat": noi_san_xuat,
        "dang_bao_che": dang_bao_che,
        "han_dung": han_dung,
        "thanh_phan_chi_tiet": thanh_phan_chi_tiet,
        "cong_dung_chi_tiet": cong_dung_chi_tiet,
        "cach_dung": cach_dung,
        "chong_chi_dinh": chong_chi_dinh,
        "tac_dung_phu": tac_dung_phu,
        "luu_y": luu_y
    }


<>:16: SyntaxWarning: invalid escape sequence '\['
<>:16: SyntaxWarning: invalid escape sequence '\['
C:\Users\Admin\AppData\Local\Temp\ipykernel_17728\3014219098.py:16: SyntaxWarning: invalid escape sequence '\['
  price_el = driver.find_element(By.CSS_SELECTOR, "body > div.bg-secondary-300 > div.w-1200px.mx-auto.min-h-screen.relative.z-1 > div > div.ml-4.flex.h-auto.w-\[540px\].cursor-default.flex-col.gap-20px > div:nth-child(1) > div:nth-child(4) > div > p")


In [53]:
def crawl_error_links(error_csv, original_csv, out_csv):
    df_error = pd.read_csv(error_csv)
    df_original = pd.read_csv(original_csv)
    
    # Merge để lấy category và drug_type từ file gốc
    df = df_error.merge(
        df_original[['link', 'category', 'drug_type']], 
        left_on='url', 
        right_on='link',
        how='left'
    )

    df = df.drop(columns=['link'])
    
    # Xử lý NaN
    # df['category'] = df['category'].fillna('')
    # df['drug_type'] = df['drug_type'].fillna('')
    driver = set_up_driver()
    results = []

    try:

        for i, row in df.iterrows():
            url = row['url']
            category = row.get('category', '')
            drug_type = row.get('drug_type', '')

            try:
                driver.get(url)
                data = get_medicine_data(driver)
                data['url'] = url
                data['category'] = category
                data['drug_type'] = drug_type
                results.append(data)
                print(f"[{i+1}/{len(df)}] crawled: {data['ten_thuoc']}")
            except Exception as e:
                results.append({'url': url, 'error': str(e)})
                print(f"[{i+1}/{len(df)}] failed: {url} with error {e}")

            time.sleep(2)  

            if (i + 1) % 50 == 0:
                pd.DataFrame(results).to_csv(out_csv, index=False, encoding='utf-8-sig')
                print(f"Saved {len(results)} records to {out_csv}")

        
    finally:
        driver.quit()
        pd.DataFrame(results).to_csv(out_csv, index=False, encoding='utf-8-sig')
        print(f"Đã lưu {len(results)} sản phẩm vào {out_csv}")

 

In [54]:
driver = set_up_driver()
driver.get("https://www.nhathuocankhang.com/thuoc-bo-va-vitamin/cezinco-110mg-5ml-h-30-ong?sku=1193668000849")
time.sleep(3)
data = get_medicine_data(driver)
print(data)
driver.quit()

{'ten_thuoc': 'Dung dịch uống Cezinco 110mg/5ml Allomed phòng ngừa và điều trị thiếu vitamin C, thiếu kẽm hộp 30 ống', 'gia': '11.300₫', 'hinh_anh': None, 'cong_dung_tom_tat': 'Phòng ngừa và điều trị thiếu vitamin C, thiếu kẽm', 'thanh_phan_chinh': 'Kẽm, Acid ascorbic', 'doi_tuong': None, 'thuong_hieu': 'Dược phẩm Allomed', 'noi_san_xuat': 'Việt Nam', 'dang_bao_che': 'Dung dịch uống', 'han_dung': '24 tháng kể từ ngày sản xuất', 'thanh_phan_chi_tiet': 'Công thức cho 5ml dung dịch\nAcid ascorbic (dưới dạng natri ascorbat) 100 mg.Kẽm sulfat monohydrat (tương đương 10mg nguyên tố kẽm) 27,44 mg.\nTá dược vừa đủ 5ml (glycerin, propylen glycol, acid citric, methyl paraben, propyl paraben, sucralose, sunset yellow dye, tartrazin yellow dye, natri metabisulfit, hương cam, nước tinh khiết).', 'cong_dung_chi_tiet': 'Phòng ngừa và điều trị thiếu vitamin C và/hoặc thiếu kẽm.', 'cach_dung': '- Cách dùng\nDùng đường uống cùng với thức ăn.\n- Liều dùng\nTrẻ 1-3 tuổi: 2,5 ml/ngày, tương đương 1/2 muỗng

In [55]:
crawl_error_links("ankhang_medicines_error_rows.csv","ankhang_medicine_links.csv", "ankhang_medicines_fix.csv")

[1/66] crawled: Dung dịch uống Cezinco 110mg/5ml Allomed phòng ngừa và điều trị thiếu vitamin C, thiếu kẽm hộp 30 ống
[2/66] crawled: Dung dịch uống Cezinco 110mg/5ml Allomed phòng ngừa và điều trị thiếu vitamin C, thiếu kẽm hộp 30 ống
[3/66] crawled: Magne - B6 Stella Tablet điều trị hạ magnesium (5 vỉ x 10 viên)
[4/66] crawled: Magne - B6 Stella Tablet điều trị hạ magnesium (5 vỉ x 10 viên)
[5/66] failed: https://www.nhathuocankhang.com/thuoc-tri-ung-thu/zoladex-inj-10-8mg?sku=1193692000032 with error Message: no such element: Unable to locate element: {"method":"xpath","selector":"//*[@id='contentRef']/div/div[3]/button"}
  (Session info: chrome=145.0.7632.45); For documentation on this error, please visit: https://www.selenium.dev/documentation/webdriver/troubleshooting/errors#nosuchelementexception
Stacktrace:
Symbols not available. Dumping unresolved backtrace:
	0x7ff725a84325
	0x7ff725a84380
	0x7ff72581d49d
	0x7ff7258782fe
	0x7ff72587860c
	0x7ff7258c8b67
	0x7ff7258c5728
	0x7ff72

In [56]:
ankhang_medicince_fix = pd.read_csv("ankhang_medicines_fix.csv")
print(ankhang_medicince_fix.shape)

(66, 20)


In [57]:
ankhang_medicince_fix.head(10)

,ten_thuoc,gia,hinh_anh,cong_dung_tom_tat,thanh_phan_chinh,doi_tuong,thuong_hieu,noi_san_xuat,dang_bao_che,han_dung,thanh_phan_chi_tiet,cong_dung_chi_tiet,cach_dung,chong_chi_dinh,tac_dung_phu,luu_y,url,category,drug_type,error
0,Dung dịch uống Cezinco 110mg/5ml Allomed phòng...,11.300₫,NaN,"Phòng ngừa và điều trị thiếu vitamin C, thiếu kẽm","Kẽm, Acid ascorbic",NaN,Dược phẩm Allomed,Việt Nam,Dung dịch uống,24 tháng kể từ ngày sản xuất,Công thức cho 5ml dung dịch\nAcid ascorbic (dư...,Phòng ngừa và điều trị thiếu vitamin C và/hoặc...,- Cách dùng\nDùng đường uống cùng với thức ăn....,Bệnh nhân quá mẫn với bất cứ thành phần nào củ...,"Vitamin C\nTăng oxalat niệu, buồn nôn hoặc nôn...",NaN,https://www.nhathuocankhang.com/thuoc-bo-va-vi...,NaN,NaN,NaN
1,Dung dịch uống Cezinco 110mg/5ml Allomed phòng...,11.300₫,NaN,"Phòng ngừa và điều trị thiếu vitamin C, thiếu kẽm","Kẽm, Acid ascorbic",NaN,Dược phẩm Allomed,Việt Nam,Dung dịch uống,24 tháng kể từ ngày sản xuất,Công thức cho 5ml dung dịch\nAcid ascorbic (dư...,Phòng ngừa và điều trị thiếu vitamin C và/hoặc...,- Cách dùng\nDùng đường uống cùng với thức ăn....,Bệnh nhân quá mẫn với bất cứ thành phần nào củ...,"Vitamin C\nTăng oxalat niệu, buồn nôn hoặc nôn...",NaN,https://www.nhathuocankhang.com/thuoc-bo-va-vi...,NaN,NaN,NaN
2,Magne - B6 Stella Tablet điều trị hạ magnesium...,90.000₫,"https://img.tgdd.vn/imgt/ankhang/f_webp,fit_ou...","Điều trị hạ magnesium huyết nặng, các rối loạn...",NaN,Người lớn và trẻ em,Stella,Việt Nam,Viên nén bao phim tan trong ruột,36 tháng kể từ ngày sản xuất,Thành phần hoạt chất:\n\nVitamin B6 (Pyridoxin...,"Điều trị hạ magnesium huyết nặng, riêng biệt h...",- Cách dùng\nMagne - B6 STELLA Tablet được uốn...,Mẫn cảm với bất kỳ thành phần nào của thuốc.\n...,Tăng magnesium huyết ít gặp sau khi uống các m...,NaN,https://www.nhathuocankhang.com/thuoc-bo-va-vi...,NaN,NaN,NaN
3,Magne - B6 Stella Tablet điều trị hạ magnesium...,90.000₫,"https://img.tgdd.vn/imgt/ankhang/f_webp,fit_ou...","Điều trị hạ magnesium huyết nặng, các rối loạn...",NaN,Người lớn và trẻ em,Stella,Việt Nam,Viên nén bao phim tan trong ruột,36 tháng kể từ ngày sản xuất,Thành phần hoạt chất:\n\nVitamin B6 (Pyridoxin...,"Điều trị hạ magnesium huyết nặng, riêng biệt h...",- Cách dùng\nMagne - B6 STELLA Tablet được uốn...,Mẫn cảm với bất kỳ thành phần nào của thuốc.\n...,Tăng magnesium huyết ít gặp sau khi uống các m...,NaN,https://www.nhathuocankhang.com/thuoc-bo-va-vi...,NaN,NaN,NaN
4,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,https://www.nhathuocankhang.com/thuoc-tri-ung-...,NaN,NaN,Message: no such element: Unable to locate ele...
5,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,https://www.nhathuocankhang.com/thuoc-bo-va-vi...,NaN,NaN,Message: no such element: Unable to locate ele...
6,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,https://www.nhathuocankhang.com/thuoc-bo-va-vi...,NaN,NaN,Message: no such element: Unable to locate ele...
7,Vitamin A-D HDpharma điều trị tình trạng thiếu...,NaN,NaN,"Dự phòng và điều trị thiếu vitamin A - D, rối ...","Vitamin D3, Vitamin A",Thuốc kê đơn - Sử dụng theo chỉ định của Bác sĩ,HDPHARMA,Việt Nam,Viên nang mềm,24 tháng kể từ ngày sản xuất,Mỗi viên nang mềm chứa\nHoạt chất:\nVitamin A ...,Dự phòng và điều trị các triệu chứng do thiếu ...,- Cách dùng\nUống trong hoặc ngay sau khi ăn.\...,Mẫn cảm với một trong các thành phần của thuốc...,Dùng Vitamin A-D với liều chỉ định thường khôn...,NaN,https://www.nhathuocankhang.com/thuoc-bo-va-vi...,NaN,NaN,NaN
8,Vitamin A-D HDpharma điều trị tình trạng thiếu...,NaN,NaN,"Dự phòng và điều trị thiếu vitamin A - D, rối ...","Vitamin D3, Vitamin A",Thuốc kê đơn - Sử dụng theo chỉ định của Bác sĩ,HDPHARMA,Việt Nam,Viên nang mềm,24 tháng kể từ ngày sản xuất,Mỗi viên nang mềm chứa\nHoạt chất:\nVitamin A ...,Dự phòng và điều trị các triệu chứng do thiếu ...,- Cách dùng\nUống trong hoặc ngay sau khi ăn.\...,Mẫn cảm với một trong các thành phần của thuốc...,Dùng Vitamin A-D